In [1]:
import pandas as pd
import requests

from pathlib import Path

In [2]:
url = "https://raw.githubusercontent.com/anilbhaila/llm-zoomcamp-finalproject/refs/heads/main/data/Ecommerce_FAQ_Chatbot_dataset.json"


In [3]:
def load_data(*args, **kwargs):
    """
    Extract data from URL. 
    
    """
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        json_data = response.json()

        faqs = json_data.get("questions")
        # Create a DataFrame
        df = pd.DataFrame(list(faqs))

        return df
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None

In [5]:
df = load_data()
df.head()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [6]:
import re


In [7]:
def transform(data: pd.DataFrame, *args, **kwargs):
    """
    Template code for a transformer block to add Chunk.

    """
    # Specify your transformation logic here

    rowNumber = 0
    documents = []

    for _, row in data.iterrows():
        number = str(rowNumber)
        rowNumber+=1
        question = str(row['question'])
        answer = str(row['answer'])

        sanitized_question = re.sub(r'\W', '_', question[:30]).lower()
        document_id = f"doc_{number}_{sanitized_question}"

        # Format the document string
        chunk = '\n'.join([
            f'question:\n{question}\n',
            f'answer:\n{answer}\n',
        ])

        documents.append({
            'chunk': chunk,
            'data': {
                'number': number,
                'question': question,
                'answer': answer
            },
            'document_id': document_id,
        })

    print(f'Documents: {len(documents)}')

    return documents

In [8]:
chunk_documents = transform(df)

Documents: 79


In [9]:
chunk_documents[0]

{'chunk': "question:\nHow can I create an account?\n\nanswer:\nTo create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n",
 'data': {'number': '0',
  'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 'document_id': 'doc_0_how_can_i_create_an_account_'}

In [10]:
from typing import Dict, List
import spacy

def transform(documents: List[Dict], *args, **kwargs):
    """
    Template code for a transformer block to Lemmatize.

    Add more parameters to this function if this block has multiple parent blocks.
    There should be one parameter for each output variable from each parent block.

    Args:
        data: The output from the upstream parent block
        args: The output from any additional upstream blocks (if applicable)

    Returns:
        Anything (e.g. data frame, dictionary, array, int, str, etc.)
    """
    count = len(documents)
    print('Documents', count)

    nlp = spacy.load('en_core_web_sm')
    
    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        if idx % 100 == 0:
            print(f'{idx + 1}/{count}')

        # Process the text chunk using spacy
        chunk = document['chunk']
        doc = nlp(chunk)
        tokens = [token.lemma_ for token in doc]

        data.append(
            dict(
                chunk=chunk,
                document_id=document_id,
                tokens=tokens,
                question=document['data']['question'],
                answer=document['data']['answer'],
            )
        )

    print('\nData', len(data))

    return data

In [11]:
lemmatize_documents = transform(chunk_documents)
lemmatize_documents[0]

Documents 79
1/79

Data 79


{'chunk': "question:\nHow can I create an account?\n\nanswer:\nTo create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n",
 'document_id': 'doc_0_how_can_i_create_an_account_',
 'tokens': ['question',
  ':',
  '\n',
  'how',
  'can',
  'I',
  'create',
  'an',
  'account',
  '?',
  '\n\n',
  'answer',
  ':',
  '\n',
  'to',
  'create',
  'an',
  'account',
  ',',
  'click',
  'on',
  'the',
  "'",
  'sign',
  'up',
  "'",
  'button',
  'on',
  'the',
  'top',
  'right',
  'corner',
  'of',
  'our',
  'website',
  'and',
  'follow',
  'the',
  'instruction',
  'to',
  'complete',
  'the',
  'registration',
  'process',
  '.',
  '\n'],
 'question': 'How can I create an account?',
 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."}

In [12]:
from typing import Dict, List

import numpy as np
import spacy

def transform(documents: List[Dict], *args, **kwargs) ->List[Dict]:
    """
    Template code for a transformer block to create embeddings.

    Add more parameters to this function if this block has multiple parent blocks.
    There should be one parameter for each output variable from each parent block.

    Args:
        data: The output from the upstream parent block
        args: The output from any additional upstream blocks (if applicable)

    Returns:
        Anything (e.g. data frame, dictionary, array, int, str, etc.)
    """
    # Specify your transformation logic here
    count = len(documents)
    print('Documents', count)

    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        if idx % 100 == 0:
            print(f'{idx + 1}/{count}')
        nlp = spacy.load('en_core_web_sm')
        tokens = document['tokens']
    
        # Combine tokens back into a single string of text used for embedding
        text = ' '.join(tokens)
        doc = nlp(text)
    
        # Average the word vectors in the doc to get a general embedding
        embedding = np.mean([token.vector for token in doc], axis=0).tolist()
    
        data.append(dict(
            chunk=document['chunk'],
            document_id=document['document_id'],
            question=document['question'],
            answer=document['answer'],
            embedding=embedding,
        ))

    return data

In [13]:
embedding_documents = transform(lemmatize_documents)
embedding_documents[0]

Documents 79
1/79


{'chunk': "question:\nHow can I create an account?\n\nanswer:\nTo create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n",
 'document_id': 'doc_0_how_can_i_create_an_account_',
 'question': 'How can I create an account?',
 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
 'embedding': [-0.21343107521533966,
  -0.5658838152885437,
  0.08929453790187836,
  -0.021446753293275833,
  -0.11743324995040894,
  0.04470497742295265,
  0.2153482884168625,
  0.03439966216683388,
  -0.037103597074747086,
  0.1286245882511139,
  -0.0429355725646019,
  0.14869722723960876,
  0.13001644611358643,
  0.21767203509807587,
  0.5346853137016296,
  0.08476872742176056,
  -0.1593695729970932,
  -0.1102224662899971,
  0.2835747301578522,
  0.0744214653968811,
  0.02511248178780079,
  0.49407

In [14]:
import json

from typing import Dict, List, Union

import numpy as np
from elasticsearch import Elasticsearch

def export_data(documents: List[Dict[str, Union[Dict, List[int], str]]], *args, **kwargs):
    """
    Exports data to some source.

    Args:
        data: The output from the upstream parent block
        args: The output from any additional upstream blocks (if applicable)

    Output (optional):
        Optionally return any object and it'll be logged and
        displayed when inspecting the block run.
    """
    # Specify your data exporting logic here
    connection_string = kwargs.get('connection_string', 'http://localhost:9200')
    index_name = kwargs.get('index_name', 'documents')
    number_of_shards = kwargs.get('number_of_shards', 1)
    number_of_replicas = kwargs.get('number_of_replicas', 0)
    dimensions = kwargs.get('dimensions')

    if dimensions is None and len(documents) > 0:
        document = documents[0]
        dimensions = len(document.get('embedding') or [])

    es_client = Elasticsearch(connection_string)

    print(f'Connecting to Elasticsearch at {connection_string}')

    index_settings = {
            "settings": {
                "number_of_shards": number_of_shards,
                "number_of_replicas": number_of_replicas,
            },
            "mappings": {
                "properties": {
                    "chunk": {"type": "text"},
                    "document_id": {"type": "text"},
                    "question": {"type": "text"},
                    "answer": {"type": "text"},
                    "embedding": {
                        "type": "dense_vector", 
                        "dims": 96,
                        "index": True,
                        "similarity": "cosine"
                    },
                }
            }
        }
    
    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)
        print(f'Index {index_name} deleted')

    es_client.indices.create(index=index_name, body=index_settings)
    print('Index created with properties:')
    print(json.dumps(index_settings, indent=2))
    print('Embedding dimensions:', dimensions)

    count = len(documents)
    print(f'Indexing {count} documents to Elasticsearch index {index_name}')
    for idx, document in enumerate(documents):
        if idx % 100 == 0:
            print(f'{idx + 1}/{count}')

        if isinstance(document['embedding'], np.ndarray):
            document['embedding'] = document['embedding'].tolist()

        es_client.index(index=index_name, document=document)

    return [d['embedding'] for d in documents[:5]]

In [18]:
embedding_indexed = export_data(embedding_documents)
embedding_indexed

Connecting to Elasticsearch at http://localhost:9200
Index created with properties:
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0
  },
  "mappings": {
    "properties": {
      "chunk": {
        "type": "text"
      },
      "document_id": {
        "type": "text"
      },
      "question": {
        "type": "text"
      },
      "answer": {
        "type": "text"
      },
      "embedding": {
        "type": "dense_vector",
        "dims": 96,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
}
Embedding dimensions: 96
Indexing 79 documents to Elasticsearch index documents
1/79


[[-0.21343107521533966,
  -0.5658838152885437,
  0.08929453790187836,
  -0.021446753293275833,
  -0.11743324995040894,
  0.04470497742295265,
  0.2153482884168625,
  0.03439966216683388,
  -0.037103597074747086,
  0.1286245882511139,
  -0.0429355725646019,
  0.14869722723960876,
  0.13001644611358643,
  0.21767203509807587,
  0.5346853137016296,
  0.08476872742176056,
  -0.1593695729970932,
  -0.1102224662899971,
  0.2835747301578522,
  0.0744214653968811,
  0.02511248178780079,
  0.4940701723098755,
  0.11377029865980148,
  -0.0924975648522377,
  0.3497902750968933,
  -0.07383574545383453,
  0.2835349142551422,
  0.011244055815041065,
  0.09735128283500671,
  0.2033621072769165,
  -0.009408318437635899,
  0.14112089574337006,
  0.29277804493904114,
  -0.21818038821220398,
  0.22006221115589142,
  -0.036354970186948776,
  0.02638719417154789,
  -0.07717086374759674,
  -0.033911097794771194,
  -0.10444222390651703,
  -0.12649233639240265,
  0.2698240876197815,
  0.2567797601222992,
  -0

In [22]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents: 79
